In [2]:
!git clone https://github.com/lukemelas/EfficientNet-PyTorch.git
%cd EfficientNet-PyTorch

Cloning into 'EfficientNet-PyTorch'...
remote: Enumerating objects: 665, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 665 (delta 21), reused 18 (delta 18), pack-reused 633 (from 2)
Receiving objects: 100% (665/665), 1.14 MiB | 4.45 MiB/s, done.
Resolving deltas: 100% (342/342), done.
/content/EfficientNet-PyTorch


In [4]:
from google.colab import files
uploaded = files.upload()

Saving img.jpg to img.jpg


In [7]:
import os
img_name = list(uploaded.keys())[0]
print(f"📷 업로드된 이미지: {img_name}")

📷 업로드된 이미지: img.jpg


In [10]:
# classify 코드 수정
from efficientnet_pytorch import EfficientNet
from PIL import Image
import torch
from torchvision import transforms

# 모델 로딩 (EfficientNet-B0)
model = EfficientNet.from_pretrained('efficientnet-b0')
model.eval()

# 이미지 불러오기 및 전처리
image = Image.open('img.jpg').convert('RGB')
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # 평균, 표준편차 정규화
                         std=[0.229, 0.224, 0.225]),
])
image_tensor = transform(image).unsqueeze(0)  # (1, 3, 224, 224)

# 추론
with torch.no_grad():
    outputs = model(image_tensor)
    probs = torch.nn.functional.softmax(outputs[0], dim=0)
    top5_probs, top5_idxs = torch.topk(probs, 5)

# ImageNet 클래스 이름 txt 로딩
!wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt -O imagenet_classes.txt
with open("imagenet_classes.txt") as f:
    labels = [line.strip() for line in f.readlines()]

# 결과 출력
print("\n Top 5 예측 결과:")
for i in range(5):
    print(f"{labels[top5_idxs[i]]}: {top5_probs[i].item()*100:.2f}%")


Loaded pretrained weights for efficientnet-b0
--2025-06-30 04:46:17--  https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10472 (10K) [text/plain]
Saving to: ‘imagenet_classes.txt’

imagenet_classes.tx 100%[===================>]  10.23K  --.-KB/s    in 0.001s  

2025-06-30 04:46:17 (9.53 MB/s) - ‘imagenet_classes.txt’ saved [10472/10472]


🔍 Top 5 예측 결과:
giant panda: 83.49%
brown bear: 0.62%
lesser panda: 0.60%
ice bear: 0.44%
Arctic fox: 0.34%


In [14]:
# B0 ~ B8 모델 추론시간, 결과 비교
from efficientnet_pytorch import EfficientNet
from PIL import Image
import torch
from torchvision import transforms
import time

# ImageNet 클래스 이름 txt 로딩
!wget https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt -O imagenet_classes.txt
with open("imagenet_classes.txt") as f:
    labels = [line.strip() for line in f.readlines()]


# B0 ~ B8 모델 정의
model_settings = {
    'efficientnet-b0' : 224,
    'efficientnet-b1' : 240,
    'efficientnet-b2' : 260,
    'efficientnet-b3' : 300,
    'efficientnet-b4' : 380,
    'efficientnet-b5' : 456,
    'efficientnet-b6' : 528,
    'efficientnet-b7' : 600,

}

# 이미지 불러옴
original_image = Image.open(img_name).convert('RGB')

# 결과 저장
results = []

# 각 모델에 대해 반복
for model_name, input_size in model_settings.items():
  print(f"\n 모델 : {model_name}, 입력 해상도 : {input_size} X {input_size}")

  model = EfficientNet.from_pretrained(model_name)
  model.eval()

  transform = transforms.Compose([
      transforms.Resize(input_size),
      transforms.CenterCrop(input_size),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
  image_tensor = transform(original_image).unsqueeze(0)

  # 각 모델별로 걸린 추론 시간 측정
  start_time = time.time()
  with torch.no_grad():
        outputs = model(image_tensor)
        probs = torch.nn.functional.softmax(outputs[0], dim=0)
        top1_prob, top1_idx = torch.topk(probs, 1)
  elapsed = time.time() - start_time

    # 결과 출력
  label = labels[top1_idx[0]]
  prob = top1_prob[0].item()
  print(f"추론 시간: {elapsed:.2f}초")
  print(f"Top-1 예측: {label} ({prob*100:.2f}%)")

  results.append((model_name, elapsed, label, prob))

# 최종 정리
print("\n 각 모델별 요약 결과:")
for name, t, label, prob in results:
    print(f"{name} | {t:.2f}s | {label} ({prob*100:.2f}%)")

--2025-06-30 05:03:57--  https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10472 (10K) [text/plain]
Saving to: ‘imagenet_classes.txt’

imagenet_classes.tx 100%[===================>]  10.23K  --.-KB/s    in 0.003s  

2025-06-30 05:03:57 (3.59 MB/s) - ‘imagenet_classes.txt’ saved [10472/10472]


 모델 : efficientnet-b0, 입력 해상도 : 224 X 224
Loaded pretrained weights for efficientnet-b0
추론 시간: 0.07초
Top-1 예측: giant panda (83.49%)

 모델 : efficientnet-b1, 입력 해상도 : 240 X 240
Loaded pretrained weights for efficientnet-b1
추론 시간: 0.10초
Top-1 예측: giant panda (87.06%)

 모델 : efficientnet-b2, 입력 해상도 : 260 X 260
Loaded pretrained weights for efficientnet-b2
추론 시간: 0.14초
Top-1 예측: giant panda (85.33%)

 모